In [1]:
import pandas as pd
import numpy as np

# ML & DL
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

# XAI
import shap

# CV
import face_recognition
import cv2

# Graph (network fraud)
import networkx as nx

# QR
import qrcode

# Utils
import hashlib

c:\Users\Pratham Gajjar\anaconda3\envs\cnn_env\lib\site-packages\face_recognition_models\__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [2]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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"   # 🔥 paste new key here
)

In [3]:
np.random.seed(42)

n = 500

df = pd.DataFrame({
    "id_verified": np.random.choice([0,1], n, p=[0.2,0.8]),
    "face_match_score": np.random.randint(30, 100, n),
    "location_changes": np.random.randint(0, 10, n),
    "previous_employers": np.random.randint(0, 5, n),
    "complaints": np.random.randint(0, 6, n),
    "work_gap_days": np.random.randint(0, 200, n),
    "age": np.random.randint(18, 60, n)
})

df["risk_label"] = (
    (df["id_verified"] == 0) |
    (df["face_match_score"] < 50) |
    (df["complaints"] > 3) |
    (df["location_changes"] > 5)
).astype(int)

In [4]:
X = df.drop("risk_label", axis=1)
y = df["risk_label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

ml_model = GradientBoostingClassifier()
ml_model.fit(X_train, y_train)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [5]:
dl_model = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500)
dl_model.fit(X_train, y_train)

,hidden_layer_sizes,"(64, ...)"
,activation,'relu'
,solver,'adam'
,alpha,0.0001
,batch_size,'auto'
,learning_rate,'constant'
,learning_rate_init,0.001
,power_t,0.5
,max_iter,500
,shuffle,True
,random_state,None


In [6]:
def get_trust_score(worker):
    df_input = pd.DataFrame([worker])
    
    ml_prob = ml_model.predict_proba(df_input)[0][1]
    dl_prob = dl_model.predict_proba(df_input)[0][1]
    
    final_prob = (ml_prob + dl_prob) / 2
    
    return int((1 - final_prob) * 100)

In [7]:
def profile_check(worker):
    issues = []
    
    if worker["work_gap_days"] > 150:
        issues.append("Long unexplained work gap")
    
    if worker["previous_employers"] == 0:
        issues.append("No employment history")
    
    return issues

In [8]:
def education_check(age, education):
    if age < 18 and education == "Graduate":
        return "Education mismatch"
    return "Valid"

In [9]:
def address_check(worker):
    if worker["location_changes"] > 5:
        return "High address volatility"
    return "Stable"

In [10]:
# ================================
# 🧠 6. FRAUD DETECTION
# ================================

def behavior_risk(worker):
    score = 0
    
    if worker["complaints"] > 3:
        score += 1
    if worker["location_changes"] > 5:
        score += 1
    
    return "Suspicious" if score >= 2 else "Normal"


def document_check(text):
    if "XXXX" in text or "0000" in text:
        return "Fake Document Suspected"
    return "Valid"


def compliance_check(worker):
    if worker["id_verified"] == 0:
        return "Legal Violation: No ID"
    return "Compliant"

In [11]:
G = nx.Graph()

def add_connection(w1, w2):
    G.add_edge(w1, w2)

def network_check(worker_id):
    return list(G.neighbors(worker_id)) if worker_id in G else []

In [12]:
# ================================
# 📷 8. FACE + DUPLICATE
# ================================

face_db = []

def multi_identity_check(enc):
    for e in face_db:
        if np.linalg.norm(e - enc) < 0.5:
            return "Duplicate Identity"
    face_db.append(enc)
    return "Unique"


def verify_face(id_img, selfie_img):
    if face_recognition is None:
        return True, 0.3  # demo fallback
    
    try:
        img1 = face_recognition.load_image_file(id_img)
        img2 = face_recognition.load_image_file(selfie_img)

        enc1 = face_recognition.face_encodings(img1)[0]
        enc2 = face_recognition.face_encodings(img2)[0]

        distance = face_recognition.face_distance([enc1], enc2)[0]
        return distance < 0.6, distance
    except:
        return False, 1.0

In [13]:
# ================================
# 🔐 9. QR + LOGGING
# ================================

def generate_qr(worker_id):
    qr = qrcode.make(worker_id)
    path = f"{worker_id}.png"
    qr.save(path)
    return path


def log_event(event):
    return hashlib.sha256(event.encode()).hexdigest()

In [14]:
# ================================
# 🤖 10. AGENTS
# ================================

def verification_agent(worker):
    if worker["id_verified"] == 0:
        return "❌ ID not verified"
    if worker["face_match_score"] < 50:
        return "⚠️ Face mismatch"
    return "✅ Verified"


def alert_agent(worker):
    alerts = []
    
    if worker["complaints"] > 3:
        alerts.append("High complaints")
    if worker["location_changes"] > 5:
        alerts.append("Suspicious movement")
    
    return alerts


def decision_agent(score):
    if score > 75:
        return "Approve"
    elif score > 50:
        return "Manual Review"
    return "Reject"

def fraud_score(worker):
    score = 0
    
    if worker["id_verified"] == 0:
        score += 50
    if worker["face_match_score"] < 50:
        score += 30
    if worker["complaints"] > 3:
        score += 20
        
    return score

In [15]:
# ================================
# FEATURES
# ================================

# 1. Digital Passport
def generate_worker_profile(result, worker_id):
    return {
        "Worker ID": worker_id,
        "Trust Score": result["Trust Score"],
        "Risk Level": result["Risk Level"],
        "Decision": result["Decision"]
    }


# 2. Employer Feedback
def update_from_feedback(score, rating):
    if rating < 3:
        score -= 10
    elif rating > 4:
        score += 5
    return max(0, min(100, score))


# 3. Anomaly Detection
def anomaly_check(worker):
    if worker["location_changes"] > 7 and worker["complaints"] > 3:
        return "Anomalous behavior detected"
    return "Normal"


# 4. Real-time Alert
def real_time_alert(worker):
    if worker["id_verified"] == 0:
        return "🚨 Immediate Action Required"
    return "No critical alerts"


# 5. Authority View
def authority_view(result, worker_id):
    return {
        "Worker ID": worker_id,
        "Risk": result["Risk Level"],
        "Fraud Score": result["Fraud Score"],
        "Action": result["Decision"]
    }


# 6. Score Explanation
def explain_score(worker):
    reasons = []
    if worker["complaints"] > 3:
        reasons.append("High complaints")
    if worker["location_changes"] > 5:
        reasons.append("Frequent movement")
    return reasons


# 7. Risk Color
def risk_color(score):
    if score > 70:
        return "Green"
    elif score > 40:
        return "Yellow"
    return "Red"


# 8. Govt DB Check
def govt_db_check(worker):
    if worker["id_verified"] == 1:
        return "Matched with Govt Records"
    return "No match found"

In [16]:
def qwen_explain(worker, result):

    prompt = f"""
    You are an AI government officer.

    Worker Details:
    {worker}

    System Result:
    {result}

    Explain:
    - Is worker safe or risky?
    - Why?
    - What action should be taken?
    """

    try:
        response = client.chat.completions.create(
            model="qwen/qwen-2.5-7b-instruct",
            messages=[{"role": "user", "content": prompt}]
        )

        return response.choices[0].message.content

    except Exception as e:
        return f"LLM error: {str(e)}"

In [17]:
!pip install pytesseract pillow

In [18]:
# ================================
# 📄 DOCUMENT VERIFICATION SYSTEM
# ================================

import pytesseract
from PIL import Image
import re

def extract_text(image_path):
    try:
        text = pytesseract.image_to_string(Image.open(image_path))
        return text
    except:
        return ""

def aadhar_check(text):
    matches = re.findall(r"\b\d{12}\b", text)
    if matches:
        return "Valid Aadhaar format detected"
    return "No Aadhaar detected"

def fake_document_check(text):
    suspicious_keywords = ["xxxx", "0000", "sample", "fake"]

    for word in suspicious_keywords:
        if word in text.lower():
            return "⚠️ Suspicious Document"

    if len(text.strip()) < 20:
        return "⚠️ Low text content (possible fake)"

    return "Document looks genuine"

def document_verification(image_path):
    text = extract_text(image_path)

    return {
        "Extracted Text": text[:100],
        "Aadhaar Check": aadhar_check(text),
        "Document Authenticity": fake_document_check(text)
    }

In [19]:
# ================================
# 🧠 11. XAI (SHAP)
# ================================

explainer = shap.Explainer(ml_model, X)

def explain_worker(worker):
    df_input = pd.DataFrame([worker])
    return explainer(df_input)

In [20]:
def full_system(worker, worker_id="W123", education="Graduate", rating=4):

    score = get_trust_score(worker)

    # Apply feedback
    score = update_from_feedback(score, rating)

    base_result = {
        "Verification": verification_agent(worker),
        "Trust Score": score,
        "Risk Level": "Low" if score > 70 else "Medium" if score > 40 else "High",
        "Fraud Score": fraud_score(worker),
        "Behavior": behavior_risk(worker),
        "Profile Issues": profile_check(worker),
        "Education Check": education_check(worker.get("age", 25), education),
        "Address Status": address_check(worker),
        "Compliance": compliance_check(worker),
        "Network": network_check(worker_id),
        "Network Risk": "High" if len(network_check(worker_id)) > 0 else "Low",
        "Alerts": alert_agent(worker),
        "Decision": decision_agent(score),
        "QR": generate_qr(worker_id),
        "Log": log_event(str(worker))
    }

    # 🔥 ADD NEW FEATURES HERE
    base_result["Risk Timeline"] = [score, score-5, score-10, score-15]
    base_result["Explanation"] = qwen_explain(worker, base_result)
    base_result["Anomaly"] = anomaly_check(worker)
    base_result["Real-Time Alert"] = real_time_alert(worker)
    base_result["Govt DB"] = govt_db_check(worker)
    base_result["Risk Color"] = risk_color(score)

    # Derived views
    base_result["Digital Passport"] = generate_worker_profile(base_result, worker_id)
    base_result["Authority View"] = authority_view(base_result, worker_id)

    # 📄 Document Verification
    doc_result = document_verification("sample_id.png")   # 👈 change later in UI
    base_result["Document Check"] = doc_result

    if "Suspicious" in doc_result["Document Authenticity"]:
        base_result["Decision"] = "Reject"

    return base_result

In [21]:
worker_safe = {
    "id_verified": 1,
    "face_match_score": 90,
    "location_changes": 1,
    "previous_employers": 3,
    "complaints": 0,
    "work_gap_days": 10,
    "age": 30
}

full_system(worker_safe)

{'Verification': '✅ Verified',
 'Trust Score': 96,
 'Risk Level': 'Low',
 'Fraud Score': 0,
 'Behavior': 'Normal',
 'Profile Issues': [],
 'Education Check': 'Valid',
 'Address Status': 'Stable',
 'Compliance': 'Compliant',
 'Network': [],
 'Network Risk': 'Low',
 'Alerts': [],
 'Decision': 'Approve',
 'QR': 'W123.png',
 'Log': '7183775ec4a6b00448e81a61a6ecd704eb53f07abb58bf334dc9082f066c7f57',
 'Risk Timeline': [96, 91, 86, 81],
 'Explanation': 'Based on the provided information for the worker and the system result, the worker can be assessed as follows:\n\n### Worker Safety Assessment\n\n#### Is the Worker Safe or Risky?\n**Safe**\n\n#### Why?\n- **Verification**: The worker\'s ID has been verified (`id_verified` = 1), indicating that the identification document matches the worker.\n- **Face Match Score**: The face match score is 90, indicating that the facial recognition technology has matched the worker\'s face with their identification document with high accuracy.\n- **Location Ch

In [22]:
worker_risky = {
    "id_verified": 0,
    "face_match_score": 40,
    "location_changes": 8,
    "previous_employers": 0,
    "complaints": 5,
    "work_gap_days": 180,
    "age": 22
}

full_system(worker_risky)

{'Verification': '❌ ID not verified',
 'Trust Score': 0,
 'Risk Level': 'High',
 'Fraud Score': 100,
 'Behavior': 'Suspicious',
 'Profile Issues': ['Long unexplained work gap', 'No employment history'],
 'Education Check': 'Valid',
 'Address Status': 'High address volatility',
 'Compliance': 'Legal Violation: No ID',
 'Network': [],
 'Network Risk': 'Low',
 'Alerts': ['High complaints', 'Suspicious movement'],
 'Decision': 'Reject',
 'QR': 'W123.png',
 'Log': '9aa7723c4a7878833c4799b6e34854446722902190dd514e52f100af702c5a3e',
 'Risk Timeline': [0, -5, -10, -15],
 'Explanation': "Based on the provided details and the system result, the worker presents significant risks that outweigh any potential benefits. Here’s a detailed explanation:\n\n### Worker Safety Assessment:\n- **Worker is considered risky**.\n- **Risks include**: Lack of ID verification, history of suspicious behavior, high number of complaints, long unexplained gaps in employment, and high address volatility.\n\n### Reasoni

In [23]:
worker_mid = {
    "id_verified": 1,
    "face_match_score": 60,
    "location_changes": 5,
    "previous_employers": 1,
    "complaints": 2,
    "work_gap_days": 100,
    "age": 24
}

full_system(worker_mid)

{'Verification': '✅ Verified',
 'Trust Score': 55,
 'Risk Level': 'Medium',
 'Fraud Score': 0,
 'Behavior': 'Normal',
 'Profile Issues': [],
 'Education Check': 'Valid',
 'Address Status': 'Stable',
 'Compliance': 'Compliant',
 'Network': [],
 'Network Risk': 'Low',
 'Alerts': [],
 'Decision': 'Manual Review',
 'QR': 'W123.png',
 'Log': '239377b593346d4bad60a06467c6a4d45ada55ed1e370c217d1157f7d1555d06',
 'Risk Timeline': [55, 50, 45, 40],
 'Explanation': "Based on the provided worker details and system result, we can evaluate the worker's safety and risk level as follows:\n\n### Safety Assessment:\n- **Is the Worker Safe or Risky?**\n  - The worker is classified as having a **medium risk level** rather than being outright unsafe.\n  \n- **Why?**\n  - **Face Match Score:** The face match score of 60 out of 100 suggests some uncertainty in identity verification, although there is a verified status indicating that the worker’s ID has been checked.\n  - **Location Changes:** There have bee